# 02 · Entrenar — MobileNetV2 con transfer learning

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Salvadrn/isic-pi/blob/main/notebooks/02_entrenar.ipynb)

Sesión 2 del proyecto **isic-pi**. Lee los arreglos que dejó el notebook 01 en tu Drive y entrena el clasificador.

**Qué hace este notebook**
1. Copia los datos de Drive al disco local de Colab (leer de Drive directamente es lento).
2. Arma el modelo: escalado de píxeles → MobileNetV2 (preentrenada en ImageNet) → capa final de 8 clases.
3. Entrena en dos fases: primero solo la capa final; luego afina las últimas capas de MobileNetV2 con un paso de aprendizaje más chico.
4. Compensa el desbalance con **pesos por clase** y decide cuándo parar mirando el **recall promedio por clase** en validación, no el accuracy (con NV dominando, el accuracy engaña).
5. Guarda el modelo en Drive (`Mi unidad/isic-pi/modelo/`) para el notebook 03.

**Entorno de ejecución:** GPU (T4). *Entorno de ejecución → Cambiar tipo de entorno → GPU.*

**Requisito:** haber corrido el notebook 01 (necesita `isic-pi/datos/` en tu Drive).

El conjunto de **test no se usa aquí**: se reserva para la evaluación final del notebook 03.

> Primera vez: pon `MODO_PRUEBA = True` (usa `datos_prueba/` del notebook 01 y una época por fase) para comprobar que todo corre antes de esperar el entrenamiento completo.

## 1. Configuración

In [ ]:
MODO_PRUEBA = False       # True: usa datos_prueba/ del notebook 01 y 1 época por fase (solo para comprobar que corre)
FORZAR = False            # True: reentrenar aunque ya exista un modelo guardado
USAR_DRIVE = True
CARPETA_DRIVE = "isic-pi"

BATCH = 64
EPOCAS_CABEZA = 6         # fase 1: backbone congelado, solo se entrena la capa final
EPOCAS_AJUSTE = 14        # fase 2: se descongelan las últimas capas de MobileNetV2
LR_CABEZA = 1e-3
LR_AJUSTE = 1e-4          # más chico: son pesos que ya sirven y solo se afinan
CAPAS_A_AJUSTAR = 40      # capas finales de MobileNetV2 que se descongelan en la fase 2
DROPOUT = 0.3
POTENCIA_PESOS = 0.5      # 0 = sin pesos por clase · 0.5 = raíz del balanceado (punto medio) · 1 = balanceado completo
PACIENCIA = 4             # épocas seguidas sin mejorar el recall promedio antes de parar la fase
SEMILLA = 42

## 2. Entorno, Drive y datos

In [ ]:
import json, os, shutil, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(SEMILLA)
if MODO_PRUEBA:
    EPOCAS_CABEZA = EPOCAS_AJUSTE = 1

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", [g.name for g in gpus] or "ninguna")
if not gpus and not MODO_PRUEBA:
    raise RuntimeError("No hay GPU. Entorno de ejecución → Cambiar tipo de entorno → GPU (T4). En CPU tardaría horas.")

In [ ]:
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / CARPETA_DRIVE
elif EN_COLAB:
    BASE = Path("/content") / CARPETA_DRIVE
else:
    BASE = Path.cwd() / CARPETA_DRIVE              # fuera de Colab (pruebas locales)

DATOS = BASE / ("datos_prueba" if MODO_PRUEBA else "datos")        # lo que dejó el notebook 01
SALIDA = BASE / ("modelo_prueba" if MODO_PRUEBA else "modelo")     # lo que deja este notebook
TRABAJO = (Path("/content") if EN_COLAB else Path.cwd()) / "trabajo" / DATOS.name   # disco local, rápido

if not (DATOS / "meta.json").exists():
    raise FileNotFoundError(f"No encuentro {DATOS / 'meta.json'}. Corre primero el notebook 01 (con el mismo MODO_PRUEBA).")
if (SALIDA / "entrenamiento.json").exists() and not FORZAR:
    raise RuntimeError(f"Ya hay un modelo entrenado en {SALIDA}. Pasa al notebook 03, o pon FORZAR = True para reentrenar.")
SALIDA.mkdir(parents=True, exist_ok=True)
TRABAJO.mkdir(parents=True, exist_ok=True)

meta = json.loads((DATOS / "meta.json").read_text())
CLASES, TAMANO = meta["clases"], meta["tamano"]
K = len(CLASES)
print(f"Datos: {DATOS}\nClases ({K}): {CLASES} | tamaño {TAMANO}×{TAMANO} | preparados el {meta['fecha']}")

In [ ]:
def traer(nombre):
    origen, destino = DATOS / nombre, TRABAJO / nombre
    if origen.resolve() == destino.resolve():
        return destino
    if destino.exists() and destino.stat().st_size == origen.stat().st_size:
        return destino
    t0 = time.time()
    shutil.copy2(origen, destino)
    print(f"{nombre}: {destino.stat().st_size / 1e9:.2f} GB copiados en {time.time() - t0:.0f} s")
    return destino


for nombre in ["train_x.npy", "train_y.npy", "split.npz", "clases.txt"]:   # test_x.npy no se trae a propósito
    traer(nombre)

X = np.load(TRABAJO / "train_x.npy", mmap_mode="r")
y = np.load(TRABAJO / "train_y.npy").astype(np.int32)
sp = np.load(TRABAJO / "split.npz")
idx_tr, idx_val = np.sort(sp["idx_tr"]), np.sort(sp["idx_val"])
assert X.shape[1:] == (TAMANO, TAMANO, 3) and len(X) == len(y)
assert (TRABAJO / "clases.txt").read_text().split() == CLASES
print(f"train {len(idx_tr):,} | val {len(idx_val):,}")

## 3. Pesos por clase

NV tiene muchísimas más fotos que DF o VASC; sin compensar, el modelo aprende a decir "nevo" casi siempre y aun así parece bueno. Cada foto pesa `(N / (K · n_clase)) ** POTENCIA_PESOS` en la pérdida.

In [ ]:
conteo = np.bincount(y[idx_tr], minlength=K)
assert conteo.min() > 0, "alguna clase no tiene imágenes en train"
pesos = (len(idx_tr) / (K * conteo)) ** POTENCIA_PESOS
class_weight = {i: float(p) for i, p in enumerate(pesos)}

display(pd.DataFrame({
    "train": conteo,
    "val": np.bincount(y[idx_val], minlength=K),
    "peso": pesos.round(2),
}, index=CLASES))

## 4. Datos para Keras

Un generador que lee lotes directo del arreglo en disco. El entrenamiento los baraja en cada época; validación va en orden fijo para poder comparar con `y_val`.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def crear_dataset(indices, barajar):
    indices = np.sort(np.asarray(indices))

    def gen():
        orden = np.random.permutation(indices) if barajar else indices
        for ini in range(0, len(orden), BATCH):
            lote = np.sort(orden[ini:ini + BATCH])
            yield X[lote], y[lote]

    firma = (tf.TensorSpec((None, TAMANO, TAMANO, 3), tf.uint8), tf.TensorSpec((None,), tf.int32))
    ds = tf.data.Dataset.from_generator(gen, output_signature=firma)
    # Se declara cuántos lotes hay por época: sin esto Keras no conoce el tamaño y avisa "input ran out of data"
    ds = ds.apply(tf.data.experimental.assert_cardinality(-(-len(indices) // BATCH)))
    return ds.map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


ds_tr = crear_dataset(idx_tr, barajar=True)
ds_val = crear_dataset(idx_val, barajar=False)
ds_val_x = ds_val.map(lambda a, b: a)          # solo imágenes, para predecir
y_val = y[idx_val]

## 5. El modelo

- **`modelo`** es lo que se exporta: píxeles 0–255 → escalado a [-1, 1] (lo que espera MobileNetV2) → MobileNetV2 → promedio global → dropout → capa de `K` clases con softmax. El escalado va **dentro** del modelo, así la Raspberry manda los píxeles tal cual.
- **`modelo_entrenamiento`** le antepone aumento de datos (giros, rotaciones, zoom, brillo, contraste). Comparte las mismas capas y pesos, pero el aumento no forma parte del modelo exportado.

In [ ]:
def construir_modelo():
    entrada = keras.Input(shape=(TAMANO, TAMANO, 3), name="imagen")
    x = layers.Rescaling(scale=1 / 127.5, offset=-1.0, name="escalado")(entrada)
    base = keras.applications.MobileNetV2(
        input_shape=(TAMANO, TAMANO, 3), include_top=False, weights="imagenet"
    )
    base.trainable = False
    x = base(x, training=False)                      # BatchNorm siempre en modo inferencia
    x = layers.GlobalAveragePooling2D(name="promedio")(x)
    x = layers.Dropout(DROPOUT, name="dropout")(x)
    salida = layers.Dense(K, activation="softmax", name="clase")(x)
    return keras.Model(entrada, salida, name="isic_mobilenetv2"), base


modelo, base = construir_modelo()

aumento = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.5, fill_mode="reflect"),
    layers.RandomZoom(0.15, fill_mode="reflect"),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
    layers.RandomContrast(0.15),
], name="aumento")
modelo_entrenamiento = keras.Sequential(
    [keras.Input(shape=(TAMANO, TAMANO, 3)), aumento, modelo], name="entrenamiento"
)
print(f"Parámetros totales: {modelo.count_params():,} | capas de MobileNetV2: {len(base.layers)}")

## 6. Entrenar

`RecallMacro` calcula, al terminar cada época, el recall de cada clase en validación y su promedio. Ese promedio manda: decide cuándo bajar el paso de aprendizaje, cuándo parar, y qué pesos conservar (los de la mejor época de cada fase).

In [ ]:
from sklearn.metrics import recall_score


class RecallMacro(keras.callbacks.Callback):
    # Los mejores pesos se guardan en un archivo (no en una lista en memoria): el orden de get_weights()
    # puede cambiar al congelar/descongelar capas y mezclaría los vectores de BatchNorm sin avisar.
    # Se guardan los del modelo interno (sin compilar), así el archivo no arrastra el estado del optimizador.
    def __init__(self, ds_x, y_true, k, ruta_pesos, modelo):
        super().__init__()
        self.ds_x, self.y_true, self.k = ds_x, y_true, k
        self.ruta_pesos = str(ruta_pesos)
        self.modelo = modelo
        self.valores = []
        self.mejor, self.mejor_epoca = -1.0, -1

    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.ds_x, verbose=0).argmax(axis=1)
        por_clase = recall_score(self.y_true, pred, labels=list(range(self.k)), average=None, zero_division=0)
        actual = float(por_clase.mean())
        logs["val_recall_macro"] = actual
        self.valores.append(actual)
        if actual > self.mejor:
            self.mejor, self.mejor_epoca = actual, epoch
            self.modelo.save_weights(self.ruta_pesos)

    def on_train_end(self, logs=None):
        if self.mejor_epoca >= 0:
            self.modelo.load_weights(self.ruta_pesos)
            print(f"→ pesos restaurados de la mejor época ({self.mejor_epoca + 1}): recall promedio {self.mejor:.3f}")


historial = []        # una entrada por época, con la fase a la que pertenece


def entrenar_fase(nombre, epocas, lr):
    entrenables = sum(int(np.prod(w.shape)) for w in modelo_entrenamiento.trainable_weights)
    print(f"\n=== Fase «{nombre}»: {epocas} épocas máx., lr {lr:g}, {entrenables:,} parámetros entrenables ===")
    modelo_entrenamiento.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    rec = RecallMacro(ds_val_x, y_val, K, TRABAJO / f"mejor_{nombre}.weights.h5", modelo)
    h = modelo_entrenamiento.fit(
        ds_tr,
        validation_data=ds_val,
        epochs=epocas,
        class_weight=class_weight,
        shuffle=False,         # el generador ya baraja en cada época
        callbacks=[
            rec,       # primero: añade val_recall_macro a los logs que ven los demás
            keras.callbacks.EarlyStopping(monitor="val_recall_macro", mode="max", patience=PACIENCIA),
            keras.callbacks.ReduceLROnPlateau(monitor="val_recall_macro", mode="max", factor=0.5,
                                              patience=2, min_lr=1e-6, verbose=1),
        ],
        verbose=2,
    )
    for i in range(len(rec.valores)):
        entrada = {k: float(v[i]) for k, v in h.history.items() if k != "val_recall_macro"}
        historial.append({"fase": nombre, **entrada, "val_recall_macro": rec.valores[i]})
    return rec


t_inicio = time.time()

# Fase 1: el backbone queda congelado, solo aprende la capa final
base.trainable = False
rec1 = entrenar_fase("cabeza", EPOCAS_CABEZA, LR_CABEZA)

In [ ]:
# Fase 2: se descongelan las últimas capas de MobileNetV2 (menos las BatchNorm) y se afina con lr chico
base.trainable = True
for capa in base.layers[:-CAPAS_A_AJUSTAR]:
    capa.trainable = False
for capa in base.layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False

rec2 = entrenar_fase("ajuste", EPOCAS_AJUSTE, LR_AJUSTE)

# Si afinar no mejoró lo que ya había logrado la fase 1, se queda con la fase 1
if rec2.mejor < rec1.mejor:
    modelo.load_weights(rec1.ruta_pesos)
    fase_elegida, rec_final = "cabeza", rec1
    print(f"\nEl ajuste fino no mejoró ({rec2.mejor:.3f} < {rec1.mejor:.3f}): se conserva la fase 1.")
else:
    fase_elegida, rec_final = "ajuste", rec2
duracion = time.time() - t_inicio
print(f"\nFase elegida: {fase_elegida} | recall promedio en validación: {rec_final.mejor:.3f} | {duracion / 60:.1f} min en total")

## 7. Cómo quedó (en validación)

Recall, precisión y F1 por clase, con los pesos elegidos. Ojo: validación es chica para las clases raras (DF y VASC tienen unas decenas de fotos), así que sus números se mueven mucho de una corrida a otra. La medición seria es el test del notebook 03.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

prob = modelo.predict(ds_val_x, verbose=0)
pred = prob.argmax(axis=1)
reporte = classification_report(y_val, pred, labels=list(range(K)), target_names=CLASES, digits=3, zero_division=0)
print(reporte)

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))
epocas = np.arange(1, len(historial) + 1)
cambio = sum(1 for h in historial if h["fase"] == "cabeza") + 0.5
ejes[0].plot(epocas, [h["loss"] for h in historial], label="train")
ejes[0].plot(epocas, [h["val_loss"] for h in historial], label="validación")
ejes[0].set_title("Pérdida")
ejes[1].plot(epocas, [h["val_recall_macro"] for h in historial], label="recall promedio (val)", color="tab:green")
ejes[1].plot(epocas, [h["val_accuracy"] for h in historial], label="accuracy (val)", color="tab:gray", linestyle="--")
ejes[1].set_title("Validación")
for e in ejes:
    e.axvline(cambio, color="k", linestyle=":", linewidth=1)
    e.set_xlabel("época (la línea punteada separa fase 1 y fase 2)")
    e.legend()
plt.tight_layout()
plt.savefig(TRABAJO / "curvas.png", dpi=130)
plt.show()

ConfusionMatrixDisplay.from_predictions(
    y_val, pred, labels=list(range(K)), display_labels=CLASES, normalize="true",
    cmap="Blues", values_format=".2f",
)
plt.title("Matriz de confusión en validación (cada fila suma 1)")
plt.tight_layout()
plt.savefig(TRABAJO / "confusion_val.png", dpi=130)
plt.show()

## 8. Guardar en Drive

In [ ]:
modelo.save(str(TRABAJO / "modelo.keras"))
(TRABAJO / "reporte_val.txt").write_text(reporte)
shutil.copy2(TRABAJO / "clases.txt", TRABAJO / "clases_modelo.txt")

try:
    gpu_nombre = tf.config.experimental.get_device_details(gpus[0]).get("device_name") if gpus else None
except Exception:
    gpu_nombre = None

recall_por_clase = recall_score(y_val, pred, labels=list(range(K)), average=None, zero_division=0)
info = {
    "fecha": time.strftime("%Y-%m-%d %H:%M"),
    "modo_prueba": MODO_PRUEBA,
    "fase_elegida": fase_elegida,
    "recall_promedio_val": float(recall_por_clase.mean()),
    "recall_por_clase_val": {c: float(r) for c, r in zip(CLASES, recall_por_clase)},
    "accuracy_val": float((pred == y_val).mean()),
    "duracion_min": round(duracion / 60, 1),
    "hiperparametros": {
        "batch": BATCH, "epocas_cabeza": EPOCAS_CABEZA, "epocas_ajuste": EPOCAS_AJUSTE,
        "lr_cabeza": LR_CABEZA, "lr_ajuste": LR_AJUSTE, "capas_a_ajustar": CAPAS_A_AJUSTAR,
        "dropout": DROPOUT, "potencia_pesos": POTENCIA_PESOS, "paciencia": PACIENCIA, "semilla": SEMILLA,
    },
    "pesos_por_clase": {c: round(class_weight[i], 4) for i, c in enumerate(CLASES)},
    "clases": CLASES,
    "tamano": TAMANO,
    "n_train": int(len(idx_tr)),
    "n_val": int(len(idx_val)),
    "datos_preparados": meta["fecha"],
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "gpu": gpu_nombre,
    "historial": historial,
}
(TRABAJO / "entrenamiento.json").write_text(json.dumps(info, ensure_ascii=False, indent=2))

(SALIDA / "entrenamiento.json").unlink(missing_ok=True)       # se copia al final: marca "terminado"
for nombre in ["modelo.keras", "clases_modelo.txt", "reporte_val.txt", "curvas.png", "confusion_val.png", "entrenamiento.json"]:
    destino = "clases.txt" if nombre == "clases_modelo.txt" else nombre
    shutil.copy2(TRABAJO / nombre, SALIDA / destino)

for p in sorted(SALIDA.iterdir()):
    print(f"{p.name:20s} {p.stat().st_size / 1e6:8.1f} MB")

# Comprobación: el modelo guardado en Drive da las mismas predicciones que el que está en memoria
recargado = keras.models.load_model(str(SALIDA / "modelo.keras"), compile=False)
muestra = X[idx_val[:16]].astype("float32")
assert np.allclose(recargado.predict(muestra, verbose=0), modelo.predict(muestra, verbose=0), atol=1e-5)
print("\n✓ Listo. El modelo guardado reproduce las predicciones. Siguiente: notebook 03 (evaluar en test y exportar a TFLite).")

## Qué quedó en Drive

```
Mi unidad/isic-pi/modelo/
├── modelo.keras          modelo completo (escalado + MobileNetV2 + capa final), sin aumento de datos
├── clases.txt            una clase por línea; el índice es la posición
├── entrenamiento.json    hiperparámetros, pesos por clase, historial, recall por clase (se escribe al final)
├── reporte_val.txt       precisión/recall/F1 por clase en validación
├── curvas.png
└── confusion_val.png
```

Siguiente sesión: **03 · Evaluar y exportar** (métricas honestas en test y conversión a `.tflite`).

**Si el recall de alguna clase sale muy bajo**, estas perillas están en la celda de configuración: `POTENCIA_PESOS` (más alto = más peso a las clases raras), `EPOCAS_AJUSTE` y `CAPAS_A_AJUSTAR`. Cada cambio requiere `FORZAR = True`.